# Treino longo do córtex do Apolo (Colab) — corpus grande de português

Corpus: ~182 MB (Wikipédia, frases do Tatoeba, gramática, conjugações, variações regionais, diálogos e **programação**: Python, JavaScript, SQL, terminal, Git e Docker), vocabulário de 140 caracteres (agora com `[ ] { } < > | \ `` ` ``).
Modelo: `intermediario` (~25M de parâmetros).

**Antes:** *Ambiente de execução → Alterar tipo de ambiente de execução → GPU* (T4 serve; A100/L4, se tiver, é bem mais rápido).
**Coloque no Google Drive** o arquivo `apolo_treino_grande.zip` (versão nova, 67 MB), na pasta `MyDrive/apolo/`, substituindo o antigo.

**Treino do zero:** o vocabulário mudou, então este treino NÃO continua o modelo anterior. Ele salva numa pasta nova (`cerebro_v2`), e o modelo bom de antes continua intacto em `MyDrive/apolo/cerebro/`.

O treino salva checkpoints no Drive a cada 2000 passos. Se a sessão cair, rode as células 1 a 3 de novo e depois a célula de **retomar**.

## 1. GPU e Google Drive

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA (ative a GPU antes de continuar)")

from google.colab import drive
drive.mount("/content/drive")

## 2. Descompactar o corpus

In [ ]:
import os
ZIP = "/content/drive/MyDrive/apolo/apolo_treino_grande.zip"
assert os.path.exists(ZIP), f"Não achei {ZIP}. Coloque o zip no Drive, em MyDrive/apolo/."
!rm -rf /content/apolo && mkdir -p /content/apolo && unzip -q -o "{ZIP}" -d /content/apolo
%cd /content/apolo
!ls -la . textos | head -12

## 3. Configurar o treino
- `PASSOS`: com `LOTE=64` e contexto 256, cada passo vê ~16 mil caracteres. 30 000 passos ≈ 500 milhões de caracteres (cerca de 3 voltas no corpus).
  Comece com um valor menor (ex.: 3000) para ver o tempo por passo no log, e só depois aumente.
- Os checkpoints (`cortex.pt` e `config.json`) vão para o Drive, em `MyDrive/apolo/cerebro_v2/`.

In [ ]:
TAMANHO = "intermediario"
PASSOS = 30000
LOTE = 64
LR = 6e-4
PASTA = "/content/drive/MyDrive/apolo/cerebro_v2"
os.makedirs(PASTA, exist_ok=True)

## 4. Treinar

In [ ]:
!python -u -W ignore cortex_v0.py treinar --dados textos/corpus_portugues.txt --tamanho {TAMANHO} \
  --passos {PASSOS} --lote {LOTE} --lr {LR} --avaliar_cada 250 --salvar_cada 2000 --pasta "{PASTA}"

### (Só se a sessão caiu) Retomar a partir do último checkpoint
Continua com os pesos salvos no Drive. A taxa de aprendizado recomeça do início, então use um número de passos menor (ex.: 10000).

In [ ]:
RESTANTES = 10000
!python -u -W ignore cortex_v0.py treinar --dados textos/corpus_portugues.txt --tamanho {TAMANHO} \
  --passos {RESTANTES} --lote {LOTE} --lr 3e-4 --avaliar_cada 250 --salvar_cada 2000 --pasta "{PASTA}" --retomar

## 5. Testar
O Apolo manda a pergunta seguida de uma quebra de linha; o córtex escreve a continuação.

In [ ]:
import cortex_v0
cortex_v0.PASTA = PASTA
from cortex_v0 import carregar, continuar

modelo, tk, dev = carregar()
for pergunta in ["Oi", "Oi, tudo bem?", "Bom dia.", "Qual é o seu nome?", "O que é um substantivo?",
                 "Como se conjuga o verbo falar no presente?", "O que significa \"uai\"?", "Qual é a capital de Rondônia?",
                 "Como mostrar uma mensagem na tela em Python?", "Como criar uma lista em Python?",
                 "Como filtrar linhas com WHERE?", "Como criar uma branch no Git?", "O que é um array em JavaScript?"]:
    texto = continuar(modelo, tk, dev, pergunta + "\n", 160, temperatura=0.5)
    print(f"> {pergunta}\n{texto.split(chr(10) * 2)[0]}\n")

## 6. Baixar os arquivos
Coloque os dois em `apolo/cerebro/` (o `cerebro_v2` é o modelo novo; o antigo está em `cerebro_backup/v1_intermediario/`) e avise o Claude Code. Eles também ficam no seu Drive.

In [ ]:
from google.colab import files
files.download(f"{PASTA}/cortex.pt")
files.download(f"{PASTA}/config.json")